# 实验二：安居客房源采集与 MySQL 存储

采集 http://www.bspider.top/anjuke/ 的房源信息，保存 CSV 后写入 MySQL `smartpark_lab.houseinfo`。

In [1]:
import csv, time
from pathlib import Path
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup

url = "http://www.bspider.top/anjuke/"
headers = {"User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) "
                          "Chrome/126.0.0.0 Safari/537.36"),
           "Referer": "http://www.bspider.top/"}
r = requests.get(url, headers=headers, timeout=30)
r.raise_for_status()
r.encoding = r.apparent_encoding
soup = BeautifulSoup(r.text, "html.parser")
print("状态码:", r.status_code, "| 标题:", soup.title.get_text())

状态码: 200 | 标题: 重庆二手房房产网，重庆二手房出售，重庆买房购房交易信息 - 58安居客


## 1. 网页分析与字段解析

Chrome F12 → Elements → Copy selector 得到的完整路径很长且含 `:nth-child()`，不稳定。
向上找到刚好包住一条房源的父容器 `div.property`（同时含标题、楼盘、地址、属性、价格），
改用**相对选择器**在容器内部找字段，更稳。

In [2]:
rows = soup.select("div.property")
print("本页房源条数:", len(rows))
row = rows[0]
node = row.select_one("h3")
print("title:", node.get_text(strip=True))
node = row.select_one("p.property-content-info-comm-name")
print("house:", node.get_text(strip=True))
# 地址文字分散在多个 span 中，get_text 拼接，不主动插入空格
node = row.select_one("p.property-content-info-comm-address")
print("address:", node.get_text(strip=True))
node = row.select_one("p.property-content-info-attribute")
print("struct:", node.get_text(strip=True))
# 属性组各项顺序需逐项打印确认，避免把朝向当楼层
texts = [p.get_text(strip=True) for p in row.select("p.property-content-info-text")]
print("属性组各项:", texts)

本页房源条数: 60
title: 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
house: 万科金域学府翰林菁华里
address: 南岸学府大道二塘路59号
struct: 3室2厅1卫
属性组各项: ['3室2厅1卫', '89.9㎡', '南北', '低层(共27层)', '2019年建造']


In [3]:
def parse_row(row):
    """从一条 div.property 解析 8 个字段：title→house→address→struct→area→floor→maketime→price"""
    node = row.select_one("h3.property-content-title-name")
    title = node.get_text(strip=True) if node else ""
    node = row.select_one("p.property-content-info-comm-name")
    house = node.get_text(strip=True) if node else ""
    node = row.select_one("p.property-content-info-comm-address")
    address = node.get_text(strip=True) if node else ""
    node = row.select_one("p.property-content-info-attribute")
    struct = node.get_text(strip=True) if node else ""
    area, floor, maketime = "", "", ""
    for t in (p.get_text(strip=True) for p in row.select("p.property-content-info-text")):
        if "㎡" in t:
            area = t                  # 面积：带 ㎡ 的项
        elif "建造" in t:
            maketime = t              # 年代：如 “2019年建造”；缺失时保持空字符串
        elif "层" in t:
            floor = t                 # 楼层：如 “低层(共27层)”，朝向“南北”不含层
    node = row.select_one("p.property-price-average")
    price = node.get_text(strip=True) if node else ""   # 单价取 元/㎡，不是万元总价
    return [title, house, address, struct, area, floor, maketime, price]

print(parse_row(rows[0]))
print(parse_row(rows[1]))
print(parse_row(rows[-1]))

['万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡']
['黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住', '两代一家', '渝北渝北黄泥磅紫荆路29号', '3室2厅2卫', '94.5㎡', '低层(共28层)', '2008年建造', '14815元/㎡']
['精装修 电梯房 4室2厅 91平', '华润凯旋天地', '南岸弹子石腾龙大道45号', '4室2厅2卫', '91.3㎡', '低层(共32层)', '2015年建造', '15886元/㎡']


## 2. 下一页与分页循环

底部“下一页”节点 class 为 `next next-active`，href 为 `?page=2` 相对地址，
用 `urljoin()` 补全。末页时按钮失去 `next-active`，作为停止条件；另设
`MAX_PAGES=30` 安全上限与已访问集合防重复/死循环。

In [4]:
nxt = soup.select_one("a.next.next-active")
href = nxt.get("href")
print("href 原值:", href)
print("urljoin 补全:", urljoin(url, href))

href 原值: ?page=2
urljoin 补全: http://www.bspider.top/anjuke/?page=2


In [5]:
MAX_PAGES = 30
current_url, visited, records, page_no = url, set(), [], 0
stop_reason = ""
while current_url:
    page_no += 1
    s = requests.get(current_url, headers=headers, timeout=30)
    s.raise_for_status(); s.encoding = s.apparent_encoding
    sp = BeautifulSoup(s.text, "html.parser")
    if "访问验证" in sp.title.get_text():
        raise RuntimeError(f"被访问验证页拦截：{current_url}")
    rs = sp.select("div.property")
    if not rs:
        raise RuntimeError(f"第{page_no}页未解析到房源，结构异常，中止")
    records.extend(parse_row(x) for x in rs)
    print(f"页码 {page_no} | {current_url} | 本页 {len(rs)} 条 | 累计 {len(records)} 条")
    visited.add(current_url)
    nxt = sp.select_one("a.next.next-active")     # 停止条件：末页按钮无 next-active
    if nxt is None:
        stop_reason = "下一页按钮无 next-active（已到末页），正常停止"; break
    href = nxt.get("href")
    if not href:
        stop_reason = "下一页无 href，结构异常，中止"; break
    nxt_url = urljoin(current_url, href)
    if nxt_url in visited or page_no >= MAX_PAGES:
        stop_reason = "达到安全页数上限" if page_no >= MAX_PAGES else "下一页地址重复，中止"
        break
    current_url = nxt_url
    time.sleep(1.2)
print("结束原因:", stop_reason)

页码 1 | http://www.bspider.top/anjuke/ | 本页 60 条 | 累计 60 条


页码 2 | http://www.bspider.top/anjuke/?page=2 | 本页 60 条 | 累计 120 条


页码 3 | http://www.bspider.top/anjuke/?page=3 | 本页 60 条 | 累计 180 条


页码 4 | http://www.bspider.top/anjuke/?page=4 | 本页 60 条 | 累计 240 条


页码 5 | http://www.bspider.top/anjuke/?page=5 | 本页 60 条 | 累计 300 条


页码 6 | http://www.bspider.top/anjuke/?page=6 | 本页 60 条 | 累计 360 条


页码 7 | http://www.bspider.top/anjuke/?page=7 | 本页 60 条 | 累计 420 条


页码 8 | http://www.bspider.top/anjuke/?page=8 | 本页 60 条 | 累计 480 条


页码 9 | http://www.bspider.top/anjuke/?page=9 | 本页 60 条 | 累计 540 条


页码 10 | http://www.bspider.top/anjuke/?page=10 | 本页 60 条 | 累计 600 条
结束原因: 下一页按钮无 next-active（已到末页），正常停止


## 3. 去重与保存 CSV

按 8 个字段完全相同去重；编码 `utf-8-sig`（Excel 可直接打开），`newline=""` 防空行。

In [6]:
FIELDS = ["title","house","address","struct","area","floor","maketime","price"]
unique, seen = [], set()
for rec in records:
    key = tuple(rec)
    if key not in seen:
        seen.add(key); unique.append(rec)
print(f"去重前 {len(records)} 条，重复 {len(records)-len(unique)} 条，去重后 {len(unique)} 条")

csv_path = Path("data/houseinfo.csv"); csv_path.parent.mkdir(exist_ok=True)
with open(csv_path, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f)
    w.writerow(FIELDS)            # 表头
    w.writerows(unique)           # 每条恰好 8 项

with open(csv_path, encoding="utf-8-sig", newline="") as f:
    rd = csv.DictReader(f)
    assert rd.fieldnames == FIELDS
    back = list(rd)
print("回读数据行数（不含表头）:", len(back))
print("首条:", back[0]); print("末条:", back[-1])

去重前 600 条，重复 102 条，去重后 498 条
回读数据行数（不含表头）: 498
首条: {'title': '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', 'house': '万科金域学府翰林菁华里', 'address': '南岸学府大道二塘路59号', 'struct': '3室2厅1卫', 'area': '89.9㎡', 'floor': '低层(共27层)', 'maketime': '2019年建造', 'price': '16465元/㎡'}
末条: {'title': '恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售', 'house': '恒大照母山山水城', 'address': '渝北照母山恒康路9号', 'struct': '3室2厅1卫', 'area': '90.2㎡', 'floor': '低层(共32层)', 'maketime': '2015年建造', 'price': '15530元/㎡'}


## 4. 读取 CSV 写入 MySQL（第三阶段，不重新爬取）

连接参数：host=`mysql`（课程容器服务名）、port=3306、db=`smartpark_lab`、
user=`student`、pwd=`123456`、charset=`utf8mb4`、autocommit=False。
用 `%s` 占位 + `executemany()` 批量插入；重复运行时按 8 字段比对库内已有
记录，只插不存在的，不清空表。出错 `rollback()`，最后先关游标再关连接。

In [7]:
conn = cur = None
try:
    with open("data/houseinfo.csv", encoding="utf-8-sig", newline="") as f:
        rd = csv.DictReader(f)
        assert rd.fieldnames == FIELDS, f"表头不符：{rd.fieldnames}"
        recs = [tuple(row[k] for k in FIELDS) for row in rd]
    print("CSV 数据行数:", len(recs))

    import pymysql
    conn = pymysql.connect(host="mysql", port=3306, database="smartpark_lab",
                           user="student", password="123456",
                           charset="utf8mb4", autocommit=False)
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM houseinfo")
    before = cur.fetchone()[0]
    print("入库前条数:", before)

    # 防重复插入：按 8 字段比对库内已有记录
    cur.execute("SELECT title,house,address,struct,area,floor,maketime,price FROM houseinfo")
    existing = {tuple(r) for r in cur.fetchall()}
    new_recs = [r for r in recs if r not in existing]
    print(f"库内已有 {len(existing)} 条，本次待插入 {len(new_recs)} 条")

    if new_recs:
        sql = ("INSERT INTO houseinfo "
               "(title,house,address,struct,area,floor,maketime,price) "
               "VALUES (%s,%s,%s,%s,%s,%s,%s,%s)")
        affected = cur.executemany(sql, new_recs)
        conn.commit()
        print(f"executemany 影响 {affected} 行，已 commit")
    else:
        print("无新增记录，跳过插入")

    cur.execute("SELECT COUNT(*) FROM houseinfo")
    after = cur.fetchone()[0]
    print(f"入库后条数: {after}，本次新增: {after-before}")
except Exception as e:
    print("出错:", e)
    if conn is not None:
        conn.rollback(); print("已 rollback")
finally:
    if cur is not None: cur.close()
    if conn is not None: conn.close()
    print("游标与连接已关闭")

CSV 数据行数: 498


入库前条数: 498
库内已有 498 条，本次待插入 0 条
无新增记录，跳过插入
入库后条数: 498，本次新增: 0
游标与连接已关闭


## 5. 验收入库结果（在 MySQL 终端执行）

```sql
SHOW DATABASES;  USE smartpark_lab;  SHOW TABLES;  DESCRIBE houseinfo;
SELECT COUNT(*) AS after_count FROM houseinfo;
SELECT title, house, address, struct, area, floor, maketime, price
FROM houseinfo ORDER BY id LIMIT 5;
```